# Multi-Layer Perceptrons Workshop — Challenge 1: TensorFlow Core API

**Course:** CSCN8010 — Prof. David Espinosa
**Team:** Sultan Atanda, John Buni, Eche Oji, Antonio Sainz
**This notebook:** Antonio Sainz (9072844) — TensorFlow Core questions

Scope, as agreed in the team task split:
1. Explain `@tf.function`, `loss_object` and `optimizer_tf`.
2. Change the batch size and record how training time changes.
3. Explain why the TensorFlow Core implementation was the fastest of the three frameworks.

The model, data preparation and training loop are the same as *Implementation 3: TensorFlow (Core API)* in `MultiLayerPerceptrons_Workshop.ipynb`. The only additions are timers and a loop over batch sizes.

## 1. Setup and data

In [1]:
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow version: {tf.__version__}")

# Same loading and preprocessing as the main workshop notebook
(x_train_tf, y_train_tf), (x_test_tf, y_test_tf) = keras.datasets.mnist.load_data()
x_train_tf = x_train_tf.astype("float32") / 255.0   # scale pixels to [0, 1]
x_test_tf = x_test_tf.astype("float32") / 255.0
x_train_flat = x_train_tf.reshape(-1, 784)          # 28x28 image -> 784-element vector
x_test_flat = x_test_tf.reshape(-1, 784)

print(f"Training data: {x_train_flat.shape}, test data: {x_test_flat.shape}")

TensorFlow version: 2.20.0
Training data: (60000, 784), test data: (10000, 784)


## 2. The TensorFlow Core model and training step

The function below wraps the workshop code so it can be re-run with different settings:
- `batch_size` — how many images are processed per gradient update.
- `use_tf_function` — whether `train_step` is compiled with `@tf.function` (the workshop default) or run eagerly, line by line in Python.

Training time is measured with `time.perf_counter()` around the training loop only (data loading and evaluation are excluded).

In [2]:
class MLP_TensorFlow(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(128, activation="relu")
        self.dense2 = tf.keras.layers.Dense(64, activation="relu")
        self.dense3 = tf.keras.layers.Dense(10)  # logits; softmax is applied inside the loss

    def call(self, x):
        x = self.dense1(x)
        x = self.dense2(x)
        return self.dense3(x)


def train_tf_core(batch_size=64, epochs=5, use_tf_function=True, verbose=True):
    """Train the workshop MLP with the TensorFlow Core API; return (training seconds, test accuracy)."""
    tf.random.set_seed(42)
    train_dataset_tf = (tf.data.Dataset.from_tensor_slices((x_train_flat, y_train_tf))
                        .shuffle(60000, seed=42).batch(batch_size))
    test_dataset_tf = tf.data.Dataset.from_tensor_slices((x_test_flat, y_test_tf)).batch(batch_size)

    model_tf = MLP_TensorFlow()
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    optimizer_tf = tf.keras.optimizers.Adam()

    def train_step(images, labels):
        with tf.GradientTape() as tape:            # records operations to compute gradients
            predictions = model_tf(images, training=True)
            loss = loss_object(labels, predictions)
        gradients = tape.gradient(loss, model_tf.trainable_variables)
        optimizer_tf.apply_gradients(zip(gradients, model_tf.trainable_variables))
        return loss

    if use_tf_function:
        train_step = tf.function(train_step)       # same effect as writing @tf.function above the def

    start = time.perf_counter()
    for epoch in range(epochs):
        epoch_loss_avg = tf.keras.metrics.Mean()
        for images, labels in train_dataset_tf:
            epoch_loss_avg.update_state(train_step(images, labels))
        if verbose:
            print(f"Epoch {epoch + 1}, Loss: {epoch_loss_avg.result():.4f}")
    train_seconds = time.perf_counter() - start

    accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()
    for images, labels in test_dataset_tf:
        accuracy_metric.update_state(labels, model_tf(images, training=False))
    return train_seconds, float(accuracy_metric.result())

### Baseline run (workshop settings: batch size 64, 5 epochs)

In [3]:
baseline_seconds, baseline_acc = train_tf_core(batch_size=64, epochs=5)
print(f"\nTraining time: {baseline_seconds:.1f} s | Test accuracy: {baseline_acc:.4f}")

Epoch 1, Loss: 0.2793
Epoch 2, Loss: 0.1147
Epoch 3, Loss: 0.0780
Epoch 4, Loss: 0.0586
Epoch 5, Loss: 0.0458

Training time: 45.6 s | Test accuracy: 0.9744


## 3. What are `@tf.function`, `loss_object` and `optimizer_tf`?

**`@tf.function`** is a decorator that converts a Python function into a TensorFlow graph. The first time `train_step` is called, TensorFlow *traces* it: it runs the Python code once, records every operation (forward pass, loss, gradients, weight update) and builds a static computation graph. Every later call executes that compiled graph directly in TensorFlow's C++ runtime instead of going back through the Python interpreter line by line. The graph can also be optimized (operations fused, memory reused), which makes each training step faster.

**`loss_object`** is an instance of `tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)`. It measures how far the model's predictions are from the true labels:
- *Sparse* — the labels are integers (0–9), not one-hot vectors.
- *Categorical cross-entropy* — the standard loss for multi-class classification; it is small when the model gives high probability to the correct digit.
- `from_logits=True` — the last layer (`dense3`) has no softmax, so the loss applies softmax internally to the raw scores (logits). This is numerically more stable than applying softmax in the model.

**`optimizer_tf`** is an instance of `tf.keras.optimizers.Adam()` (default learning rate 0.001). After the `GradientTape` computes the gradient of the loss with respect to every weight and bias, `optimizer_tf.apply_gradients(...)` uses those gradients to update the parameters. Adam adapts the step size for each parameter using running averages of past gradients, which usually makes training converge faster than plain gradient descent.

## 4. Effect of batch size on training time

Same model, 5 epochs, `@tf.function` enabled. Only the batch size changes.

In [4]:
batch_sizes = [32, 64, 128, 256]
batch_results = []
for bs in batch_sizes:
    seconds, acc = train_tf_core(batch_size=bs, epochs=5, verbose=False)
    steps_per_epoch = int(np.ceil(60000 / bs))
    batch_results.append((bs, steps_per_epoch, seconds, acc))
    print(f"Batch size {bs:>3}: {steps_per_epoch:>4} steps/epoch, training time {seconds:6.1f} s, test accuracy {acc:.4f}")

Batch size  32: 1875 steps/epoch, training time   78.6 s, test accuracy 0.9743
Batch size  64:  938 steps/epoch, training time   22.6 s, test accuracy 0.9743
Batch size 128:  469 steps/epoch, training time   28.7 s, test accuracy 0.9731
Batch size 256:  235 steps/epoch, training time   12.2 s, test accuracy 0.9715


In [5]:
import plotly.graph_objects as go

fig = go.Figure(go.Scatter(
    x=[str(r[0]) for r in batch_results],
    y=[round(r[2], 1) for r in batch_results],
    mode="lines+markers+text",
    text=[f"{r[2]:.1f} s" for r in batch_results],
    textposition="top center",
))
fig.update_layout(
    title="TensorFlow Core: training time vs batch size (5 epochs)",
    xaxis_title="Batch size",
    yaxis_title="Training time (s)",
    template="plotly_white",
    width=750, height=450,
)
fig.show()

### Results: batch size vs training time

| Batch size | Steps per epoch | Training time (5 epochs) | Test accuracy |
|---|---|---|---|
| 32 | 1,875 | 78.6 s | 97.43% |
| 64 (workshop default) | 938 | 22.6 s | 97.43% |
| 128 | 469 | 28.7 s | 97.31% |
| 256 | 235 | 12.2 s | 97.15% |

- Overall, training time fell as batch size grew: from 78.6 s at batch size 32 to 12.2 s at 256 (about 6.4x faster). Every epoch still processes all 60,000 images, but a larger batch means fewer steps (1,875 → 235), so there is less per-step overhead (Python loop, launching the graph, optimizer update) and the CPU works on larger, more efficient matrix operations.
- The trend is not perfectly smooth: batch size 128 (28.7 s) was slower than 64 (22.6 s) in this run. On a laptop CPU, other programs and power management change the available compute from one run to the next, so single timings are noisy. The baseline run at batch size 64 in Section 2 took 45.6 s with the same settings, which shows the size of this run-to-run variation. A fair comparison would repeat each setting several times and average.
- Test accuracy decreased slightly as batch size grew (97.43% → 97.15%). With the same 5 epochs, a larger batch means fewer weight updates, so the model has fewer chances to learn. The differences are a few tenths of a percent, so batch size mainly changed speed, not accuracy.
- Absolute times depend on the machine and on what else is running, which is also why they differ from the 28.4 s in the team's comparison table; the trend, not the exact seconds, is the finding.

## 5. Why was TensorFlow Core the fastest?

To isolate the effect of `@tf.function`, the same training step is run for one epoch (batch size 64) twice: once compiled as a graph and once eagerly (plain Python, operation by operation).

In [6]:
graph_seconds, graph_acc = train_tf_core(batch_size=64, epochs=1, use_tf_function=True, verbose=False)
eager_seconds, eager_acc = train_tf_core(batch_size=64, epochs=1, use_tf_function=False, verbose=False)

print(f"With @tf.function (graph): {graph_seconds:6.1f} s, test accuracy {graph_acc:.4f}")
print(f"Without (eager mode):      {eager_seconds:6.1f} s, test accuracy {eager_acc:.4f}")
print(f"Eager mode is {eager_seconds / graph_seconds:.1f}x slower")

With @tf.function (graph):    4.8 s, test accuracy 0.9561
Without (eager mode):        34.2 s, test accuracy 0.9517
Eager mode is 7.2x slower


### Answer: why TensorFlow Core was the fastest

With one epoch at batch size 64, the graph version took 4.8 s and the eager version 34.2 s (7.2x slower). Accuracy was similar (95.61% vs 95.17%), so both did the same computation; only the way it was executed changed.

- `@tf.function` traces `train_step` once and compiles the forward pass, loss, gradients and Adam update into one graph that runs in TensorFlow's C++ runtime. In eager mode, Python dispatches every operation separately on each of the 938 steps.
- The data pipeline is light: `tf.data` slices batches from arrays that were normalised and flattened once, up front.
- By comparison, the PyTorch code runs eagerly and its `DataLoader` applies the torchvision transforms (`ToTensor`, `Normalize`) to each image one at a time, every epoch. Keras `model.fit` also compiles a graph, but adds per-step work such as metric tracking and the progress bar, plus a pass over the 10% validation split at the end of every epoch.